# Use Case 8 — Industry Showcase: Healthcare, Legal, Retail, General

Four more industry-domain agents built on the same AgentTune spine APIs used in the BFSI notebook.

## 1. Healthcare — clinical triage, end to end

In [ ]:
"""
Healthcare case study — a clinical triage agent, end to end.
============================================================

A nurse-triage agent reads a presenting complaint, checks vitals and history, and assigns an
acuity level (ESI 1 = most urgent … 5 = least). Its trajectories are evaluated, and an agent
that loops on the vitals check without deciding is caught by the real FailureDetector and
turned into corrective training data. GPU-free.

NOT a medical device — an illustrative, deterministic example of the spine on clinical triage.
"""
from agenttune.agentic import (
    Project, DictToolHarness, ReActStrategy, SelfHealLoop,
    Event, EventKind, EventLog, agentic_metrics,
)
from agenttune.decide.closed_loop.contracts import ClassifiedFailure, TrainingExample

In [2]:
def check_vitals(patient=""):
    return {"patient": patient, "hr": 128, "spo2": 91, "bp": "88/60"}

In [3]:
def check_history(patient=""):
    return {"patient": patient, "conditions": ["COPD"], "allergies": ["penicillin"]}

In [4]:
TOOLS = {"check_vitals": check_vitals, "check_history": check_history}

In [5]:
def triage_policy(state):
    """vitals → history → acuity. Canned, model-free."""
    if state.step == 0:
        return {"name": "check_vitals", "arguments": {"patient": "p-77"},
                "thought": "unstable-sounding complaint — vitals first"}
    if state.step == 1:
        return {"name": "check_history", "arguments": {"patient": "p-77"},
                "thought": "low SpO2 + tachycardia; check comorbidities"}
    return {"name": "finish", "arguments": {"answer": "ESI-2"},
            "thought": "hypoxia + COPD → high acuity, ESI-2"}

In [6]:
def main():
    proj = Project(strategy=ReActStrategy(triage_policy, max_steps=5),
                   harness=DictToolHarness(TOOLS, max_steps=5))

    for complaint in ("chest tightness, short of breath", "sprained ankle, walking"):
        log = proj.infer(f"triage patient: {complaint}")
    print(f"[triage]   {len(proj.trajectories)} patients triaged, last in {len(log)} events")

    per = [agentic_metrics(l) for l in proj.trajectories]
    mean = {k: round(sum(p[k] for p in per) / len(per), 2) for k in per[0]}
    print(f"[evaluate] mean trajectory metrics -> {mean}")

    # A degraded agent loops on vitals without assigning acuity.
    stuck = EventLog(tier="light")
    for _ in range(4):
        stuck.append(Event(EventKind.TOOL_CALL,
                           {"action": {"name": "check_vitals", "arguments": {"patient": "p-9"}}}))
        stuck.append(Event(EventKind.TOOL_RESULT, {"output": "hr: 128"}))
    proj.add_trajectory(stuck)
    failures = proj.heal()
    print(f"[monitor]  FailureDetector -> {len(failures)} stuck agent(s): {[f.failure_type for f in failures]}")

    classify = lambda fs: [ClassifiedFailure(failure=f, root_cause=f.failure_type, confidence=0.9,
                                             analysis="re-checks vitals without assigning acuity") for f in fs]
    generate = lambda cfs: [TrainingExample(
        trajectory_id=c.failure.trajectory_id, original_failure_type=c.failure.failure_type,
        root_cause=c.root_cause,
        prompt=[{"role": "user", "content": "assign an acuity level"}],
        chosen=[{"role": "assistant", "content": "check history, then assign ESI-1..5"}],
        rejected=[{"role": "assistant", "content": "check vitals again"}],
    ) for c in cfs]
    summary = SelfHealLoop(classify, generate).run(failures)
    print(f"[heal]     corrective dataset -> {summary['n_dataset_rows']} row(s) "
          f"{list(summary['dataset'][0].keys())}; ready to retrain the triage agent")

In [7]:
if __name__ == "__main__":
    main()

[triage]   2 patients triaged, last in 13 events


[evaluate] mean trajectory metrics -> {'tac': 0.0, 'ter': 1.0, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0.0}
[monitor]  FailureDetector -> 1 stuck agent(s): ['loop_collapse']
[heal]     corrective dataset -> 1 row(s) ['prompt', 'chosen', 'rejected']; ready to retrain the triage agent


## 2. Healthcare — medical-coding agentic distillation

In [ ]:
"""
Healthcare case study — distil a medical-coding agent into an on-prem model.
============================================================================

Clinical text can't leave the hospital's network, and coding volume makes hosted inference
expensive. Agentic distillation fits: a strong teacher agent assigns diagnosis codes to
encounters, and its trajectories are behavior-cloned into a small (<3B) student that runs
inside the hospital's own environment. GPU-free; the injected trainer is where the real
fine-tune would run on the hospital's GPU.

Illustrative only — not a certified coding system.
"""
from agenttune.agentic import Project
from agenttune.agentic.rollout_engines.demo_engine import DemoRolloutEngine

In [9]:
class OnPremTrainerFactory:
    def __init__(self):
        self.captured = {}

    def __call__(self, *, model, train_dataset, **kwargs):
        self.captured = {"model": model, "n_rows": len(train_dataset)}

        class _Trainer:
            def train(_self):
                return {"status": "would-train-in-hospital-vpc",
                        "student": self.captured["model"], "n_rows": self.captured["n_rows"]}
        return _Trainer()

In [10]:
def main():
    teacher = Project()
    encounters = [
        "encounter: type 2 diabetes with neuropathy, routine follow-up",
        "encounter: community-acquired pneumonia, admitted",
        "encounter: essential hypertension, medication review",
    ]
    teacher.collect_rollout(DemoRolloutEngine(), encounters, tools=[], max_steps=2)
    print(f"[teacher]  {len(teacher.trajectories)} coded-encounter trajectories collected")

    rows = teacher.sft_dataset()
    print(f"[teacher]  behavior-cloning dataset -> {len(rows)} SFT rows (schema {list(rows[0].keys())})")

    factory = OnPremTrainerFactory()
    result = teacher.distill("med-coder-1.3B", trainer_factory=factory)
    print(f"[distill]  student='{factory.captured['model']}' fed {factory.captured['n_rows']} teacher rows")
    print(f"[distill]  result -> {result}")

In [11]:
if __name__ == "__main__":
    main()

[teacher]  3 coded-encounter trajectories collected
[teacher]  behavior-cloning dataset -> 3 SFT rows (schema ['messages', 'segment_weights', 'loss_mask'])
[distill]  student='med-coder-1.3B' fed 3 teacher rows
[distill]  result -> {'status': 'would-train-in-hospital-vpc', 'student': 'med-coder-1.3B', 'n_rows': 3}


/workspace/agenttune/src/agenttune/agentic/project.py:140: UserWarning: No tools provided — running in single-turn mode.
  fn = create_rollout_fn(


## 3. Legal — contract review, end to end

In [ ]:
"""
Legal case study — Contract-review agent, end to end.
=====================================================

A contract-review agent reads a clause, checks the counterparty's obligations and the governing
law, and decides ACCEPT / NEGOTIATE / REJECT. The point of the spine: those trajectories feed
both evaluation and failure-recovery. A reviewer that loops re-reading the same clause without
deciding is caught by the real FailureDetector and turned into corrective training data.
GPU-free.

Illustrative only — not legal advice.
"""
from agenttune.agentic import (
    Project, DictToolHarness, ReActStrategy, SelfHealLoop,
    Event, EventKind, EventLog, agentic_metrics,
)
from agenttune.decide.closed_loop.contracts import ClassifiedFailure, TrainingExample

In [13]:
# --- the reviewer's tools (deterministic stand-ins for a clause library / CLM lookup) ---
def read_clause(clause_id=""):
    return {"clause_id": clause_id, "type": "limitation_of_liability",
            "cap": "fees paid in prior 3 months"}

In [14]:
def check_obligation(party=""):
    return {"party": party, "indemnifies": False, "uncapped_carveouts": ["IP", "confidentiality"]}

In [15]:
def governing_law(clause_id=""):
    return {"clause_id": clause_id, "law": "Delaware", "acceptable": True}

In [16]:
TOOLS = {"read_clause": read_clause, "check_obligation": check_obligation,
         "governing_law": governing_law}

In [17]:
def reviewer_policy(state):
    """A ReAct contract reviewer: read the clause -> check obligations -> decide. Canned."""
    if state.step == 0:
        return {"name": "read_clause", "arguments": {"clause_id": "LOL-7"},
                "thought": "start with the liability cap"}
    if state.step == 1:
        return {"name": "check_obligation", "arguments": {"party": "counterparty"},
                "thought": "cap is low and one-sided; check indemnity"}
    return {"name": "finish", "arguments": {"answer": "NEGOTIATE"},
            "thought": "low mutual cap, no indemnity -> push back"}

In [18]:
def main():
    proj = Project(strategy=ReActStrategy(reviewer_policy, max_steps=5),
                   harness=DictToolHarness(TOOLS, max_steps=5))

    # 1) Review a batch of clauses (light-tier episodes).
    for clause in ("LOL-7 limitation of liability", "IND-2 indemnification"):
        log = proj.infer(f"review contract clause: {clause}")
    print(f"[review]   {len(proj.trajectories)} clauses reviewed, "
          f"last decision reached in {len(log)} events")

    # 2) Evaluate the review trajectories with the real programmatic metrics.
    per = [agentic_metrics(l) for l in proj.trajectories]
    mean = {k: round(sum(p[k] for p in per) / len(per), 2) for k in per[0]}
    print(f"[evaluate] mean trajectory metrics -> {mean}")

    # 3) A degraded reviewer slips out: it re-reads the same clause, never deciding.
    stuck = EventLog(tier="light")
    for _ in range(4):
        stuck.append(Event(EventKind.TOOL_CALL,
                           {"action": {"name": "read_clause", "arguments": {"clause_id": "LOL-7"}}}))
        stuck.append(Event(EventKind.TOOL_RESULT, {"output": "type: limitation_of_liability"}))
    proj.add_trajectory(stuck)
    failures = proj.heal()
    print(f"[monitor]  FailureDetector -> {len(failures)} stuck reviewer(s): {[f.failure_type for f in failures]}")

    # 4) Turn the failure into corrective preference data (retrain-ready).
    classify = lambda fs: [ClassifiedFailure(failure=f, root_cause=f.failure_type, confidence=0.85,
                                             analysis="re-reads the clause without deciding") for f in fs]
    generate = lambda cfs: [TrainingExample(
        trajectory_id=c.failure.trajectory_id, original_failure_type=c.failure.failure_type,
        root_cause=c.root_cause,
        prompt=[{"role": "user", "content": "review the contract clause"}],
        chosen=[{"role": "assistant", "content": "check obligations, then decide ACCEPT/NEGOTIATE/REJECT"}],
        rejected=[{"role": "assistant", "content": "read the clause again"}],
    ) for c in cfs]
    summary = SelfHealLoop(classify, generate).run(failures)
    print(f"[heal]     corrective dataset -> {summary['n_dataset_rows']} preference row(s) "
          f"{list(summary['dataset'][0].keys())}; ready to retrain the review agent")

In [19]:
if __name__ == "__main__":
    main()

[review]   2 clauses reviewed, last decision reached in 13 events
[evaluate] mean trajectory metrics -> {'tac': 0.0, 'ter': 1.0, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0.0}
[monitor]  FailureDetector -> 1 stuck reviewer(s): ['loop_collapse']
[heal]     corrective dataset -> 1 preference row(s) ['prompt', 'chosen', 'rejected']; ready to retrain the review agent


## 4. Legal — precedent retrieval over a citation graph

In [ ]:
"""
Legal case study — Precedent retrieval over a citation graph.
=============================================================

Case law is a graph: a case cites an earlier case, and a later ruling can overrule it. A vector
store recalls opinions that *read* alike; a citation graph recalls what a case actually *relies
on* — and never surfaces an unrelated matter just because the language is similar. This seeds a
small precedent graph and shows an agent recalling the authorities a case leans on. GPU-free
(no embedder, no model).

Illustrative only — not legal advice.
"""
from agenttune.agentic import GraphMemory, MemoryItem

In [21]:
def main():
    g = GraphMemory()

    # Nodes: the case at hand, the authorities it relies on, and an unrelated matter.
    subject = g.write(MemoryItem(content="Acme v. Byte (2024): trade-secret misappropriation"))
    pepsi = g.write(MemoryItem(content="PepsiCo v. Redmond (1995): inevitable disclosure"))
    uniform = g.write(MemoryItem(content="Uniform Trade Secrets Act §1: definition of a secret"))
    unrelated = g.write(MemoryItem(content="Doe v. City (2019): municipal zoning dispute"))

    # Edges: what the subject case cites as authority.
    g.link(subject, pepsi, "cites")
    g.link(subject, uniform, "cites")

    # A litigator asks: what authority does Acme v. Byte rely on?
    recalled = [i.content for i in g.read("Acme v. Byte (2024): trade-secret misappropriation", k=5)]
    print("[query]  authorities relied on by 'Acme v. Byte':")
    for c in recalled:
        print(f"           - {c}")
    print(f"[check]  zoning matter surfaced? {'Doe v. City (2019): municipal zoning dispute' in recalled}  "
          f"(should be False — uncited, unrelated)")

In [22]:
if __name__ == "__main__":
    main()

[query]  authorities relied on by 'Acme v. Byte':
           - PepsiCo v. Redmond (1995): inevitable disclosure
           - Uniform Trade Secrets Act §1: definition of a secret
[check]  zoning matter surfaced? False  (should be False — uncited, unrelated)


## 5. Retail — catalog-enrichment agentic distillation

In [ ]:
"""
Retail case study — distil a catalog-enrichment agent into a small model.
=========================================================================

Catalog enrichment is high-volume and low-margin: for every SKU an agent normalises the title,
fills attributes, and writes a description. Running a frontier model per SKU across millions of
listings is expensive. Agentic distillation fits: a strong teacher agent enriches a sample, and
its trajectories are behavior-cloned into a small (<3B) student that runs cheaply at catalog
scale. GPU-free; the injected trainer is where the real fine-tune would run on a GPU.
"""
from agenttune.agentic import Project
from agenttune.agentic.rollout_engines.demo_engine import DemoRolloutEngine

In [24]:
class CatalogTrainerFactory:
    def __init__(self):
        self.captured = {}

    def __call__(self, *, model, train_dataset, **kwargs):
        self.captured = {"model": model, "n_rows": len(train_dataset)}

        class _Trainer:
            def train(_self):
                return {"status": "would-train-on-gpu",
                        "student": self.captured["model"], "n_rows": self.captured["n_rows"]}
        return _Trainer()

In [25]:
def main():
    teacher = Project()
    skus = [
        "sku: mens running shoe, mesh, size 10, blue — raw supplier title",
        "sku: stainless water bottle 750ml — raw supplier title",
        "sku: wireless earbuds, ANC, usb-c — raw supplier title",
    ]
    teacher.collect_rollout(DemoRolloutEngine(), skus, tools=[], max_steps=2)
    print(f"[teacher]  {len(teacher.trajectories)} enriched-SKU trajectories collected")

    rows = teacher.sft_dataset()
    print(f"[teacher]  behavior-cloning dataset -> {len(rows)} SFT rows (schema {list(rows[0].keys())})")

    factory = CatalogTrainerFactory()
    result = teacher.distill("catalog-enricher-1.3B", trainer_factory=factory)
    print(f"[distill]  student='{factory.captured['model']}' fed {factory.captured['n_rows']} teacher rows")
    print(f"[distill]  result -> {result}")

In [26]:
if __name__ == "__main__":
    main()

[teacher]  3 enriched-SKU trajectories collected
[teacher]  behavior-cloning dataset -> 3 SFT rows (schema ['messages', 'segment_weights', 'loss_mask'])
[distill]  student='catalog-enricher-1.3B' fed 3 teacher rows
[distill]  result -> {'status': 'would-train-on-gpu', 'student': 'catalog-enricher-1.3B', 'n_rows': 3}


## 6. Retail — order-issue triage, evaluated

In [ ]:
"""
Retail case study — an order-issue triage agent, evaluated.
===========================================================

A post-purchase agent reads a shopper's message, checks the order, and routes it (WISMO /
REFUND / REPLACEMENT). It runs over a small labelled set and scores each routing with
`Project.evaluate`, so you see accuracy the way you would for any classifier — but the agent is
a tool-using ReAct loop, not a single prompt. GPU-free.
"""
from agenttune.agentic import Project, DictToolHarness, ReActStrategy

In [28]:
def lookup_order(order_id=""):
    return {"order_id": order_id, "status": "delivered", "carrier": "UPS"}

In [29]:
def route_for(text: str) -> str:
    t = text.lower()
    if "refund" in t or "money back" in t or "charged" in t:
        return "REFUND"
    if "broken" in t or "damaged" in t or "wrong item" in t:
        return "REPLACEMENT"
    return "WISMO"  # "where is my order"

In [30]:
def main():
    messages = [
        {"task": "where is my order, it hasn't arrived yet", "expected": "WISMO"},
        {"task": "the mug arrived damaged, it's broken", "expected": "REPLACEMENT"},
        {"task": "please refund me, I was charged twice", "expected": "REFUND"},
    ]

    def policy(state):
        # step 0: pull up the order; step 1: route on the message.
        if state.step == 0:
            return {"name": "lookup_order", "arguments": {"order_id": "o-9001"}, "thought": "check order"}
        return {"name": "finish", "arguments": {"answer": route_for(state.task)},
                "thought": "route the issue"}

    proj = Project(strategy=ReActStrategy(policy, max_steps=4),
                   harness=DictToolHarness({"lookup_order": lookup_order}, max_steps=4))

    report = proj.evaluate(messages)
    print(f"[evaluate] routed {report['n']} messages, accuracy = {report['mean_score']:.2f}")
    for m, s in zip(messages, report["scores"]):
        verdict = "correct" if s == 1.0 else "wrong"
        print(f"  [{verdict:7}] {m['expected']:12} <- {m['task'][:44]}")

In [31]:
if __name__ == "__main__":
    main()

[evaluate] routed 3 messages, accuracy = 1.00
  [correct] WISMO        <- where is my order, it hasn't arrived yet
  [correct] REPLACEMENT  <- the mug arrived damaged, it's broken
  [correct] REFUND       <- please refund me, I was charged twice


## 7. General — competitor-research agent

In [ ]:
"""
Simple case study — a competitor-research agent.
================================================

The everyday agent: given a product, search for competitors, compare them, and write a short
brief. This is the smallest useful ReAct loop — search → search → summarise → finish — and it
still produces a full `EventLog` you can evaluate, train on, or distil. GPU-free (canned tools
and policy; swap in a real web-search tool and model unchanged).
"""
from agenttune.agentic import Project, DictToolHarness, ReActStrategy, EventKind

In [33]:
# --- tools (deterministic stand-ins for a real web search) ---
_INDEX = {
    "note-taking apps": ["Notion", "Obsidian", "Roam"],
    "Notion": {"pricing": "$8/user/mo", "strength": "all-in-one workspace"},
    "Obsidian": {"pricing": "free / $8 sync", "strength": "local-first markdown"},
    "Roam": {"pricing": "$15/mo", "strength": "networked thought"},
}

In [34]:
def web_search(query=""):
    return _INDEX.get(query, f"no results for {query!r}")

In [35]:
def compare(name=""):
    return _INDEX.get(name, {"error": f"unknown {name}"})

In [36]:
TOOLS = {"web_search": web_search, "compare": compare}

In [37]:
def research_policy(state):
    """search competitors → compare the top one → finish with a brief."""
    if state.step == 0:
        return {"name": "web_search", "arguments": {"query": "note-taking apps"},
                "thought": "find the competitor set"}
    if state.step == 1:
        return {"name": "compare", "arguments": {"name": "Obsidian"},
                "thought": "compare the local-first option"}
    return {"name": "finish",
            "arguments": {"answer": "Top competitors: Notion, Obsidian, Roam. "
                                    "Obsidian is the local-first play at $8 sync."},
            "thought": "enough to brief the team"}

In [38]:
def main():
    proj = Project(strategy=ReActStrategy(research_policy, max_steps=5),
                   harness=DictToolHarness(TOOLS, max_steps=5))

    log = proj.infer("Who competes with our note-taking app, and how?")

    print(f"[run]     {len(log)} events over {sum(1 for e in log if e.kind is EventKind.TOOL_CALL)} tool calls")
    for e in log:
        if e.kind is EventKind.TOOL_CALL:
            a = e.payload.get("action", {})
            print(f"  → {a.get('name'):<11} {a.get('arguments')}")
    answer = next((e.payload['action']['arguments'].get('answer')
                   for e in log if e.kind is EventKind.TOOL_CALL
                   and e.payload.get('action', {}).get('name') == 'finish'), None)
    print(f"[brief]   {answer}")
    print(f"[spine]   this EventLog is ready for evaluate / train / distil — same as any other")

In [39]:
if __name__ == "__main__":
    main()

[run]     13 events over 3 tool calls
  → web_search  {'query': 'note-taking apps'}
  → compare     {'name': 'Obsidian'}
  → finish      {'answer': 'Top competitors: Notion, Obsidian, Roam. Obsidian is the local-first play at $8 sync.'}
[brief]   Top competitors: Notion, Obsidian, Roam. Obsidian is the local-first play at $8 sync.
[spine]   this EventLog is ready for evaluate / train / distil — same as any other


## 8. General — support-ticket triage, evaluated

In [ ]:
"""
Simple case study — a support-ticket triage agent, evaluated.
=============================================================

A customer-support agent reads a ticket, checks the account, and routes it (BILLING / BUG /
ESCALATE). It runs over a small labelled set and scores each routing with `Project.evaluate`,
so you can see accuracy the same way you would for any classifier — but the agent is a
tool-using ReAct loop, not a single prompt. GPU-free.
"""
from agenttune.agentic import Project, DictToolHarness, ReActStrategy

In [41]:
def lookup_account(user=""):
    return {"user": user, "plan": "pro", "open_tickets": 1}

In [42]:
def route_for(text: str) -> str:
    t = text.lower()
    if "refund" in t or "charge" in t or "invoice" in t:
        return "BILLING"
    if "crash" in t or "error" in t or "broken" in t:
        return "BUG"
    return "ESCALATE"

In [43]:
def main():
    tickets = [
        {"task": "I was double charged on my invoice this month", "expected": "BILLING"},
        {"task": "the export button crashes every time", "expected": "BUG"},
        {"task": "I want to speak to someone about my contract", "expected": "ESCALATE"},
    ]

    def policy(state):
        # step 0: look up the account; step 1: route based on the ticket text.
        if state.step == 0:
            return {"name": "lookup_account", "arguments": {"user": "u-42"}, "thought": "check plan"}
        return {"name": "finish", "arguments": {"answer": route_for(state.task)},
                "thought": "route the ticket"}

    proj = Project(strategy=ReActStrategy(policy, max_steps=4),
                   harness=DictToolHarness({"lookup_account": lookup_account}, max_steps=4))

    report = proj.evaluate(tickets)
    print(f"[evaluate] routed {report['n']} tickets, accuracy = {report['mean_score']:.2f}")
    for t, s in zip(tickets, report["scores"]):
        verdict = "correct" if s == 1.0 else "wrong"
        print(f"  [{verdict:7}] {t['expected']:9} <- {t['task'][:44]}")

In [44]:
if __name__ == "__main__":
    main()

[evaluate] routed 3 tickets, accuracy = 1.00
  [correct] BILLING   <- I was double charged on my invoice this mont
  [correct] BUG       <- the export button crashes every time
  [correct] ESCALATE  <- I want to speak to someone about my contract
